# Dark Manifold Virtual Cell — Full Prototype Run

Runs every prototype end-to-end and captures each one's output into a JSON file.
Code is pulled from https://github.com/Nikku03/cell (public repo, no auth needed).

**Two tracks:**
- **Cell-simulator substrate** (P0–P10): atom conservation, Syn3A biochemistry,
  learned rates, spatial simulator, LSODA integration.
- **Architecture investigation** (P11–P15): neural PDEs, unitary evolution,
  gauge/memory/rule mechanisms tested against matched-parameter baselines.

**Plus:** P8b and P8c (scaled-up permutation-invariant networks for full Syn3A).

Expected total run time on GPU Colab: ~30–45 minutes. All outputs stream to
`/content/dmvc_outputs.json`. Safe to use Runtime → Run all.

## 1. Setup

### 1.1 Install dependencies

In [ ]:
!pip install -q numpy python-libsbml scipy torch
print('deps installed')

### 1.2 Clone the Syn3A data repo (needed by P2 and onward)

In [ ]:
import os
if not os.path.exists('/content/Minimal_Cell'):
    !git clone --depth 1 https://github.com/Luthey-Schulten-Lab/Minimal_Cell.git /content/Minimal_Cell
sbml = '/content/Minimal_Cell/CME_ODE/model_data/iMB155_NoH2O.xml'
assert os.path.exists(sbml), f'SBML not found at {sbml}'
print('syn3a data ok')

### 1.3 Clone the DMVC code from GitHub

Public repo, no auth needed. Files land in `/content/dmvc/`.

In [ ]:
import os, shutil
# Clean any stale directory so we always get a fresh checkout
if os.path.exists('/content/dmvc'):
    shutil.rmtree('/content/dmvc')
!git clone --depth 1 https://github.com/Nikku03/cell.git /content/dmvc

# Verify the prototypes are present
required = [
    'prototype_p0.py', 'prototype_p1.py',
    'prototype_p2_syn3a.py', 'prototype_p2b_rebalance.py',
    'prototype_p3b_stamps.py',
    'prototype_p4_kinetics.py', 'prototype_p4b_kinetics_coupled.py',
    'prototype_p5_boundary.py', 'prototype_p6_physiological.py',
    'prototype_p7_learned_rates.py', 'prototype_p7b_tuned.py',
    'prototype_p8_perm_invariant.py',
    'prototype_p8b_full_syn3a.py', 'prototype_p8c_scaled.py',
    'prototype_p9_lsoda.py', 'prototype_p10_learned_spatial.py',
    'prototype_p11_neural_pde.py', 'prototype_p12_gauge.py',
    'prototype_p13_unitary.py', 'prototype_p14_memory.py',
    'prototype_p15_rule_discovery.py',
]
missing = [f for f in required if not os.path.exists(f'/content/dmvc/{f}')]
assert not missing, f'Missing prototype files: {missing}'
print(f'all {len(required)} prototype files present in /content/dmvc/')

### 1.4 Patch hardcoded paths

Prototypes were written with `/home/claude/dmvc/...` paths hardcoded. Redirect
every one to the Colab equivalent (SBML file, kinetics TSVs, output files).

In [ ]:
import re, os, glob

SBML = '/content/Minimal_Cell/CME_ODE/model_data/iMB155_NoH2O.xml'
DATA_BASE_SRC = '/home/claude/dmvc/data/Minimal_Cell/'
DATA_BASE_DST = '/content/Minimal_Cell/'

assert os.path.exists(SBML), f'SBML not found at {SBML}'
kinetics_check = DATA_BASE_DST + 'CME_ODE/model_data/Central_AA_Zane_Balanced_direction_fixed_nounqATP.tsv'
assert os.path.exists(kinetics_check), f'kinetics TSV not found at {kinetics_check}'

for f in sorted(glob.glob('/content/dmvc/prototype_*.py')):
    with open(f) as fh:
        src = fh.read()
    orig = src
    # Fix SBML_PATH assignment
    src = re.sub(r"SBML_PATH\s*=\s*['\"][^'\"]*['\"]", 'SBML_PATH = "' + SBML + '"', src)
    # Redirect all /home/claude/dmvc/data/Minimal_Cell/ -> /content/Minimal_Cell/
    src = src.replace(DATA_BASE_SRC, DATA_BASE_DST)
    # Fix P8c output path
    src = src.replace("'/home/claude/dmvc/p8c_results.npz'", "'/content/dmvc/p8c_results.npz'")
    if src != orig:
        with open(f, 'w') as fh:
            fh.write(src)
        print('patched', os.path.basename(f))

print('done')

### 1.5 Add /content/dmvc to Python path

In [ ]:
import sys, os
if '/content/dmvc' not in sys.path:
    sys.path.insert(0, '/content/dmvc')
os.chdir('/content/dmvc')
print('cwd:', os.getcwd())

## 2. GPU check

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__}')
print(f'device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('(CPU fallback -- P8c will be slower but should still run)')

In [ ]:
# Prototypes run as subprocesses, so they don't inherit this notebook's
# device state. P8b and P8c have built-in GPU detection; other neural
# prototypes (P7, P7b, P8, P10, P11-P15) run on CPU but are small and fast.
print('GPU will be used by: P8b, P8c (built-in cuda detection)')
print('CPU will be used by: P7, P7b, P8, P10, P11, P12, P13, P14, P15')
print('No GPU needed for: P0-P6, P9 (pure numpy/scipy)')

## 3. Output capture infrastructure

Every prototype runs through a helper that:
- captures stdout and stderr
- times the run
- writes each result into `/content/dmvc_outputs.json` incrementally

In [ ]:
import json, time, subprocess, os, sys

OUTPUTS_PATH = '/content/dmvc_outputs.json'

def load_outputs():
    if os.path.exists(OUTPUTS_PATH):
        with open(OUTPUTS_PATH) as f:
            return json.load(f)
    return {}

def save_outputs(data):
    with open(OUTPUTS_PATH, 'w') as f:
        json.dump(data, f, indent=2)

def run_prototype(name, script_path, timeout=1800):
    outputs = load_outputs()
    print(f'=== {name} ===')
    if not os.path.exists(script_path):
        entry = {
            'name': name, 'script': script_path,
            'returncode': -2, 'elapsed_s': 0.0,
            'stdout': '', 'stderr': f'script not found: {script_path}',
            'timed_out': False,
        }
        outputs[name] = entry
        save_outputs(outputs)
        print(f'[{name}] SCRIPT NOT FOUND; aborting this run')
        return entry
    print(f'running {script_path} ...')
    t0 = time.time()
    try:
        result = subprocess.run(
            [sys.executable, script_path],
            cwd='/content/dmvc',
            capture_output=True, text=True, timeout=timeout,
            env={**os.environ, 'PYTHONPATH': '/content/dmvc'})
        elapsed = time.time() - t0
        entry = {
            'name': name, 'script': script_path,
            'returncode': result.returncode, 'elapsed_s': round(elapsed, 2),
            'stdout': result.stdout, 'stderr': result.stderr,
            'timed_out': False,
        }
    except subprocess.TimeoutExpired as e:
        elapsed = time.time() - t0
        entry = {
            'name': name, 'script': script_path,
            'returncode': -1, 'elapsed_s': round(elapsed, 2),
            'stdout': (e.stdout.decode() if e.stdout else ''),
            'stderr': (e.stderr.decode() if e.stderr else '') + f'\n*** TIMED OUT after {timeout}s ***',
            'timed_out': True,
        }
    outputs[name] = entry
    save_outputs(outputs)
    tail = entry['stdout'].strip().split('\n')[-25:]
    for line in tail:
        print(line)
    print(f'[{name}] rc={entry["returncode"]} elapsed={entry["elapsed_s"]}s')
    print()
    return entry

print(f'outputs will stream to {OUTPUTS_PATH}')
# Reset any prior run
if os.path.exists(OUTPUTS_PATH):
    os.remove(OUTPUTS_PATH)
    print('cleared previous outputs')

## 4. Cell-simulator track: P0 → P10

### P0 — atom conservation via subspace projection

Smallest test. 3 atom types, random reactions, projection keeps atoms conserved.

In [ ]:
run_prototype('P0_atom_conservation', '/content/dmvc/prototype_p0.py')

### P1 — stamp+flavor embeddings, balanced reactions

In [ ]:
run_prototype('P1_stamps', '/content/dmvc/prototype_p1.py')

### P2 — load real Syn3A SBML

In [ ]:
run_prototype('P2_syn3a', '/content/dmvc/prototype_p2_syn3a.py')

### P2b — rebalance with H⁺ and H₂O

In [ ]:
run_prototype('P2b_rebalance', '/content/dmvc/prototype_p2b_rebalance.py')

### P3b — compartment-aware stamps

In [ ]:
run_prototype('P3b_compartment_stamps', '/content/dmvc/prototype_p3b_stamps.py')

### P4 — well-mixed Syn3A kinetics

In [ ]:
run_prototype('P4_kinetics', '/content/dmvc/prototype_p4_kinetics.py')

### P4b — kinetics coupled to Ψ field

In [ ]:
run_prototype('P4b_kinetics_coupled', '/content/dmvc/prototype_p4b_kinetics_coupled.py')

### P5 — boundary fluxes (open system)

In [ ]:
run_prototype('P5_boundary', '/content/dmvc/prototype_p5_boundary.py')

### P6 — physiological tuning

In [ ]:
run_prototype('P6_physiological', '/content/dmvc/prototype_p6_physiological.py', timeout=300)

### P7 — first learned rate predictor

In [ ]:
run_prototype('P7_learned_rates', '/content/dmvc/prototype_p7_learned_rates.py', timeout=300)

### P7b — tuned learned rate predictor

In [ ]:
run_prototype('P7b_tuned', '/content/dmvc/prototype_p7b_tuned.py', timeout=600)

### P8 — permutation-invariant network (original)

Scenario 1 (glycolysis held-out): should pass at ~47% zero-shot error.
Scenario 2 (full Syn3A): the original null — collapses to zero due to
rate-scale disparity. Kept here for reference.

In [ ]:
run_prototype('P8_perm_invariant', '/content/dmvc/prototype_p8_perm_invariant.py', timeout=900)

### P8b — full Syn3A scale, fixed loss design

Per-reaction log-scale normalization + species embeddings fix the collapse.
Median error ~60%, sign agreement >90%.

In [ ]:
run_prototype('P8b_full_syn3a', '/content/dmvc/prototype_p8b_full_syn3a.py', timeout=900)

### P8c — scaled-up P8b (GPU headline run)

Hidden 512, embedding 16, more samples, longer training. Uses GPU if available.
Expected on GPU: 2-5 minutes. If T1/T2 pass here the single-network Syn3A
claim has clean support.

In [ ]:
run_prototype('P8c_scaled', '/content/dmvc/prototype_p8c_scaled.py', timeout=1800)

### P8d — Diagnostic analysis of P8c error distribution

Runs the same P8c training, then performs four diagnostic analyses:
- Q1: pathological reactions (max single-sample loss)
- Q2: error by reaction class (compartment, cofactor, participant count)
- Q3: error correlation with per-reaction rate range
- Q4: what do the 15 best and 15 worst reactions have in common

Saves diagnostics to `p8d_diagnostic.npz`.

In [ ]:
run_prototype('P8d_diagnostic', '/content/dmvc/prototype_p8d_diagnostic.py', timeout=1200)

### P8e — Training stability fix (DISABLED)

Previous run's attempt to fix P8c's training instability via (gradient
clipping + warmup + y_norm clamp) made everything worse. The combination
over-regularized training and the model got stuck in a trivial "predict
near zero" plateau.

Disabled for this run while we get P8d's diagnostic output first.
If the diagnostic reveals a clear root cause, a targeted P8f will replace
this. See the P8e results in the previous run's JSON if curious.

In [ ]:
# P8e ran last time and its 'fix' made results worse (median 49.84% -> 79.93%,
# sign acc 96.64% -> 73.41%). Keeping this cell disabled until we have a fresh
# hypothesis. P8d diagnostic (above) is what we're focused on this run.
print('P8e_stable: SKIPPED. See comment above.')

### P9 — LSODA stiff integration

In [ ]:
run_prototype('P9_lsoda', '/content/dmvc/prototype_p9_lsoda.py', timeout=600)

### P10 — learned rates in spatial simulator (hybrid)

In [ ]:
run_prototype('P10_learned_spatial', '/content/dmvc/prototype_p10_learned_spatial.py', timeout=900)

## 5. Architecture track: P11 → P15

### P11 — neural PDE (Fisher-KPP)

Local conv + periodic padding learns a nonlinear reaction-diffusion PDE.
All 4 tests pass; rollout error < 1%.

In [ ]:
run_prototype('P11_neural_pde', '/content/dmvc/prototype_p11_neural_pde.py', timeout=600)

### P12 — gauge/connection field (NULL)

Baseline absorbs gauge structure implicitly; learned A collapses. Filed null.

In [ ]:
run_prototype('P12_gauge', '/content/dmvc/prototype_p12_gauge.py', timeout=600)

### P13 — structural unitary evolution (strong pass)

Schrödinger target. Structurally-unitary model beats baseline 3× on rollout.
Renormalization alone doesn't close the gap.

In [ ]:
run_prototype('P13_unitary', '/content/dmvc/prototype_p13_unitary.py', timeout=600)

### P14 — memory bank (NULL)

Retrieval attention collapsed to uniform. Task wasn't memory-hungry. Filed null.

In [ ]:
run_prototype('P14_memory', '/content/dmvc/prototype_p14_memory.py', timeout=600)

### P15 — rule discovery (NULL)

Mixture-of-K rules on multi-regime reaction-diffusion. Mode collapse;
rule-region agreement at chance. Third null in modular-factorization pattern.

In [ ]:
run_prototype('P15_rule_discovery', '/content/dmvc/prototype_p15_rule_discovery.py', timeout=600)

## 6. Run summary

In [ ]:
import json
with open('/content/dmvc_outputs.json') as f:
    outputs = json.load(f)
total_time = sum(v['elapsed_s'] for v in outputs.values())
print(f'Captured {len(outputs)} prototype runs. Total wall clock: {total_time:.1f}s')
print()
print(f'{"prototype":<28s} {"rc":>4s} {"time (s)":>10s} {"status":>10s}')
print('-' * 60)
for name, entry in outputs.items():
    rc = entry['returncode']
    status = 'OK' if rc == 0 else ('TIMEOUT' if entry.get('timed_out') else 'FAIL')
    print(f'{name:<28s} {rc:>4d} {entry["elapsed_s"]:>10.1f} {status:>10s}')

In [ ]:
# Show last 20 lines of stdout for each failed run
for name, entry in outputs.items():
    if entry['returncode'] != 0:
        print(f'=== FAILURE: {name} ===')
        tail = entry['stdout'].strip().split('\n')[-20:]
        for line in tail:
            print(line)
        if entry['stderr'].strip():
            print('--- stderr ---')
            err_tail = entry['stderr'].strip().split('\n')[-10:]
            for line in err_tail:
                print(line)
        print()

## 7. Download outputs

In [ ]:
from google.colab import files
files.download('/content/dmvc_outputs.json')

## 8. Notes

- **P8c is the headline run.** Memory-starved on the 4 GB container; should
  complete in ~5 minutes on GPU and tells us whether T1 (median < 50%) and
  T2 (> 25% of reactions under 20%) pass for the full-Syn3A single-network claim.
- **P12, P14, P15 are filed nulls.** They'll run and show non-passing results;
  that's the expected outcome, captured for reproducibility.
- The JSON is ~1-2 MB once full.